# Pandas Support: CRISP-DM de ponta a ponta

Aplicação local para apoio ao atendente, com documentação versionada, avaliação e operação. Este notebook usa somente a biblioteca padrão e lê evidências distribuídas; não executa inferência automaticamente. A aplicação web é independente dele.

**Conclusão:** engenharia reproduzida; qualidade insuficiente para atendimento autônomo. Resultados negativos são parte da análise.

In [1]:
from pathlib import Path
import json, gzip
from collections import Counter
ROOT = Path.cwd()
if not (ROOT / 'config.json').exists():
    ROOT = ROOT.parent
def load(name):
    return json.loads((ROOT/name).read_text(encoding='utf-8'))
print('Versão pandas:', load('config.json')['pandas_version'])

Versão pandas: 2.2.3


## 1. Negócio

Tarefa: ajudar um atendente a responder questões sobre leitura de arquivos, tipos e junções. Saídas: orientação com fontes, esclarecimento ou encaminhamento. Inverter cardinalidade ou converter um identificador para ausente pode comprometer a análise. Não foi medido impacto comercial; a hipótese de reduzir procura manual depende de avaliação com usuários.

## 2. Dados

30 páginas oficiais, versão 2.2.3. Benchmark sintético autoral de 120 cenários e 60 famílias. Divisão por família evita colocar uma paráfrase no desenvolvimento e a outra no teste. Referências auditadas com assistência de IA, sem revisão humana externa independente.

In [2]:
manifest = load('assets/manifest.json')
cases = load('evaluation/cases.json')
print('Páginas:', len(manifest['sources']), '| Passagens:', manifest['chunks'])
print('Tipos de caso:', dict(Counter(c['case_type'] for c in cases)))
groups = {s:{c['family'] for c in cases if c['split']==s} for s in ['development','test']}
assert groups['development'].isdisjoint(groups['test'])
print('Famílias disjuntas:', {s:len(g) for s,g in groups.items()})

Páginas: 30 | Passagens: 1068
Tipos de caso: {'answerable': 72, 'ambiguous': 18, 'out_of_scope': 12, 'false_premise': 6, 'adversarial': 12}
Famílias disjuntas: {'development': 30, 'test': 30}


## 3. Preparação

Navegação e links brutos poluíam a busca. Segmentação anterior cortava a definição de validate antes de many_to_one. A preparação preserva o conjunto de opções, condensa traceback longo e guarda proveniência. O rebuild deve reproduzir o hash do corpus congelado. O índice registra digest do encoder; modelos e corpus distintos não podem ser misturados.

In [3]:
with gzip.open(ROOT/'assets/corpus.json.gz','rt',encoding='utf-8') as f:
    corpus=json.load(f)
print('Corpus:', manifest['corpus_id'])
print('Arquivos de entrada, corpus e vetores distribuídos em assets/.')
print('Para reconstruir: python main.py rebuild')

Corpus: 81396567c3f64252
Arquivos de entrada, corpus e vetores distribuídos em assets/.
Para reconstruir: python main.py rebuild


## 4. Modelagem

BM25 é o baseline lexical. Busca semântica usa embeddinggemma via Ollama e cosseno exato; híbrida combina rankings por RRF. Escolha baseada em desenvolvimento. qwen3:8b gera orientação curta; código anexa citações reais. Planejador e verificador são chamadas do mesmo modelo e compartilham limitações.

In [4]:
retrieval = load('reports/development_retrieval.json')
for r in retrieval:
    print(r['method'], 'Hit@5=', round(r['hit_at_k']['mean'],4), 'n=',r['hit_at_k']['denominator'], 'MRR=',round(r['mrr']['mean'],4))

bm25 Hit@5= 0.5263 n= 38 MRR= 0.289
dense Hit@5= 0.7368 n= 38 MRR= 0.5272
hybrid Hit@5= 0.6579 n= 38 MRR= 0.5329


## 5. Avaliação

Separar teste de software, recuperação de evidência e correção final. Acertar a ação answer não garante o conteúdo. A revisão semântica exige correção completa, sustentação completa e ausência de erro crítico. Notas e casos estão distribuídos para inspeção.

In [5]:
final = load('reports/release_evaluation/test/results.json')
review = load('reports/semantic_summary.json')
print('Casos:',final['n'], '| Ação esperada:',final['action_matches'])
print('Respostas aprovadas:',review['approved_answers'],'/',review['emitted_answers'])
print('Resolução entre respondíveis:',review['correct_resolution_lower_bound'])
print('Mediana / P95 segundos:',round(final['latency_median_ms']/1000,2),round(final['latency_p95_ms']/1000,2))
print('Revisão humana independente:',review['independent_human_review'])

Casos: 60 | Ação esperada: 49
Respostas aprovadas: 24 / 49
Resolução entre respondíveis: 0.6
Mediana / P95 segundos: 15.45 20.74
Revisão humana independente: False


### Erros que o verificador não bloqueou

NA textual foi recomendado como na_values; nomes literais de colunas foram traduzidos; igualdade escalar de NaN foi confundida com merge de nulos; perguntas ambíguas receberam decisões sem contexto. Fontes reais e validação de JSON não impediram esses erros.

In [6]:
annotations=load('reports/review_annotations.json')['overrides']
for case_id in ['case-006-1','case-028-2','case-032-2','case-040-2']:
    print(case_id, annotations[case_id][4])

case-006-1 na_values=['NA'] converte justamente o código que deveria preservar em ausente.
case-028-2 Muda cliente_id para client_id; estratégia left_on/right_on correta, literal errado.
case-032-2 Afirma equivalência ao SQL para chaves nulas; pandas pode casar nulos.
case-040-2 Recomenda coerce sem conhecer campo; pode transformar identificadores em ausentes.


## 6. Implantação e operação

Servidor HTTP local e interface web, adaptador FastAPI opcional, logs por request_id, painel de tempos, feedback e histórico de ativação do corpus. O teste de instalação usa dados temporários e inferência real. CI está configurada; envio e execução no GitHub são etapas da publicação. Não houve hospedagem pública ou operação empresarial.

In [7]:
installation=load('reports/clean_installation.json')
print('Instalação isolada passou:',installation['passed'])
for step in installation['steps']:
    if 'question' in step:
        print(step['expected'], step['passed'])
print((ROOT/'reports/test_all.txt').read_text(encoding='utf-8').split('----------------------------------------------------------------------')[-1].strip())

Instalação isolada passou: True
answer True
handoff True
clarify True
Ran 73 tests in 4.449s

OK


## Decisão final e próxima hipótese

O projeto demonstra coleta, preparação, experimentação, avaliação, implantação local e monitoramento com evidências. A geração foi reprovada para atendimento autônomo. O atendente deve conferir as fontes e registrar feedback. A próxima iteração precisa melhorar interpretação, preservar identificadores, avaliar exemplos executáveis e usar novos casos reservados e avaliadores externos. Não alegar ganho de produtividade ou acurácia sem medir.

**Execução:** python main.py serve. **Relatório completo:** reports/RESULTADOS.md. **Runbook:** docs/OPERACAO.md.